## 1.2 GFSK 调制解调完整流程

在上一节中，我们了解了本章的学习目标和前置要求。从本节开始，我们将深入 GFSK 调制的核心原理与实现细节，系统学习从随机比特到 IQ 信号的四步调制流程，理解 AWGN 信道与 SNR 的基本概念，并最终完成 GFSK 解调和误码率统计。

本节学习大纲如下：

- 项目结构介绍
- GFSK 调制具体步骤
- AWGN 信道与 SNR
- GFSK 非相干解调
- 误码率统计

## 项目结构
在开始实验前，先了解本课程的源码结构。
```
src/nearlink_sdr/
├── common/       # 通用编码 (CRC, BCH, Polar, 码块分割, 加扰, MCS)
├── phy/          # 物理层 (调制, 帧结构, 信道, 均衡, 同步, 跳频, USRP)
├── mac/          # MAC 层 (链路管理, 信令, 安全, QoS, 广播, 调度)
└── sim/          # 仿真 (链路仿真, USRP 环回)
```

可执行以下代码查看本课程具体的源码结构：

In [ ]:
!tree ./src -L 3

---

### 1. 本实验涉及的关键文件

```
src/nearlink_sdr/
├── phy/
│   ├── gfsk.py              <- GFSKModulator: GFSK 调制 (NRZ+高斯滤波+相位积分+exp映射)
│   │                           GFSKDemodulator: 频率鉴别非相干解调
│   └── channel.py           <- ChannelModel: AWGN 信道模型 (加噪)
```


---

### 2. 什么是数字调制？

数字通信中，信息以比特（0 和 1）的形式存在。但无线电波不能直接发送 0 和 1——它只能发送连续的波形。数字调制就是建立比特和波形的映射规则。

如：**FSK（Frequency Shift Keying，频移键控）** 是一种用不同频率表示不同比特的数字调制方式：发送比特 1 时发射较高频率的载波，发送比特 0 时发射较低频率的载波。

<img src="./images/fsk_intro.png" width="600">

**相位连续的 FSK（CPFSK）** 在码元转换时刻相位保持连续，避免相位跳变带来的频谱展宽。其数学表达式为：

$$s(t) = A \cdot \cos(2\pi f_c t + 2\pi \Delta f \int_0^t m(\tau)d\tau)$$

由于相位是频率的积分，频率的改变不会导致相位的突变，信号波形始终平滑连续。其中 $A$ 为幅度，$f_c$ 为载波中心频率，$\Delta f$ 为峰值频率偏移，$m(t)$ 为双极性基带信号（0 映射为 -1，1 映射为 +1）。

**GFSK（Gaussian Frequency Shift Keying）** 就是在 CPFSK 的基础上，对基带信号增加了一级高斯低通滤波器，将尖锐跳变进一步平滑化，使发射频谱更窄。GFSK 是星闪 SLE 标准帧类型 1 采用的基础调制方式。

本次实验中 GFSK 调制的相关代码如下：

```python
    def modulate(self, bits: np.ndarray) -> np.ndarray:
        """GFSK调制。

        :param bits: 输入比特序列, shape (N,), 值为 0/1

        :returns: 复基带IQ信号, shape (N*sps,)
        """
        # NRZ映射: 0 -> -1, 1 -> +1
        nrz = 2.0 * bits.astype(float) - 1.0

        # 上采样：使用矩形脉冲（repeat），而非冲激
        upsampled = np.repeat(nrz, self.sps)

        # 高斯滤波平滑频率轨迹
        filtered = np.convolve(upsampled, self._gauss_filter, mode='same')

        # 频率积分得到瞬时相位
        # 每个符号的总相位变化应为 h*pi
        freq_deviation = self.mod_index * np.pi / self.sps
        phase = np.cumsum(filtered) * freq_deviation

        # 生成复基带信号
        signal = np.exp(1j * phase)
        return signal
```

---

### 3. 生成随机比特序列
用 NumPy 产生 10000 个随机比特。

In [ ]:
# 将本地 src/ 目录加入 Python 搜索路径
import sys
sys.path.insert(0, "../src")
import numpy as np
import matplotlib.pyplot as plt

num_bits = 10000
rng = np.random.default_rng(42)
tx_bits = rng.integers(0,2,num_bits)
print(f"生成 {num_bits} 个随机比特")

---

### 4. GFSK 调制

GFSK 的整个调制解调过程可执行以下代码查看：

In [ ]:
!cat -n src/nearlink_sdr/sim/link_sim.py | sed -n "91,128p"

GFSK 调制分四步，下面逐步拆解展示每一步的中间结果：

<img src="./images/gfsk_modulation.png" width="800">

**步骤 1: NRZ 映射 + 上采样**
比特 0/1 转为双极性电平（0->-1, 1->+1），然后用 np.repeat 每比特复制 sps 次：

In [ ]:
sps = 8
mod_index = 0.5
nrz = 2.0 * tx_bits[:20].astype(float) - 1.0
upsampled = np.repeat(nrz, sps)
print(f"NRZ 完成: {len(nrz)} 符号 -> {len(upsampled)} 采样点")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 4.5))

# 上图: NRZ 映射后的符号 (点图)
ax1.scatter(np.arange(len(nrz)), nrz, c="b", s=25, zorder=3, label="NRZ")
ax1.set_xlabel("Bit Index"); ax1.set_ylabel("Amplitude")
ax1.set_title("Step 1a: NRZ Mapping (0->-1, 1->+1)")
ax1.set_ylim(-1.5, 1.5); ax1.set_xticks(np.arange(0, len(nrz), 5))
ax1.legend(); ax1.grid(True, ls="--", alpha=0.5)

# 下图: 上采样后的采样点 (每比特复制 sps 次)
ax2.scatter(np.arange(len(upsampled)), upsampled, c="b", s=3, zorder=3, label="Upsampled")
ax2.set_xlabel("Sample Index"); ax2.set_ylabel("Amplitude")
ax2.set_title(f"Step 1b: Upsampling (repeat x{sps})")
ax2.set_ylim(-1.5, 1.5); ax2.set_xlim(0, len(upsampled)-1)
ax2.legend(); ax2.grid(True, ls="--", alpha=0.3)
for b in range(0, len(nrz) + 1):
    ax2.axvline(x=b*sps, color="gray", ls="--", alpha=0.4, lw=0.5)

plt.tight_layout(); plt.show()

**步骤 2: 高斯滤波**
把信号送进高斯低通滤波器（BT=0.5），使用 NRZ 的不再有尖锐跳变，使发射频谱更窄：

In [ ]:
from nearlink_sdr.phy.gfsk import GFSKModulator
mod_tmp = GFSKModulator(sps=sps, mod_index=mod_index)
gauss_filter = mod_tmp._gauss_filter
filtered = np.convolve(upsampled, gauss_filter, mode="same")
fig, ax = plt.subplots(figsize=(12, 3))
t_up = np.arange(len(upsampled))
ax.plot(t_up, upsampled, "b-", alpha=0.4, lw=1, label="Before")
ax.plot(t_up, filtered, "r-", lw=1.5, label="After Gaussian")
ax.set_xlabel("Sample Index"); ax.set_title("Step 2: Gaussian Filtering")
ax.set_xlim(0, len(upsampled)-1); ax.grid(True, ls="--", alpha=0.5)
ax.legend(); plt.show()

**步骤 3: 相位积分**
滤波后的值代表瞬时频率偏移量，cumsum 累计为相位：

In [ ]:
freq_dev = mod_index * np.pi / sps
phase = np.cumsum(filtered) * freq_dev
fig,ax = plt.subplots(figsize=(12,3))
ax.plot(np.arange(len(phase)),phase,'g-',lw=1.2)
ax.set_xlabel('Sample Index');ax.set_xlim(0,len(phase)-1)
ax.set_ylabel('Phase (rad)');ax.set_title('Step 3: Phase Integration')
ax.grid(True,ls='--',alpha=0.5);plt.show()

**步骤 4: 观察 GFSK 调制波形（仅用于理解，仿真中不执行）**

实际仿真直接使用复基带 IQ 信号 $e^{j\phi(t)}$信号，不涉及载波搬移。

为直观理解 GFSK 的频率调制效果，此处临时引入一个载波（$f_c = 0.5$ 周期/符号），生成观察信号 $s(t) = \cos(2\pi f_c t + \phi(t))$，将相位变化转化为肉眼可见的频率疏密——NRZ=+1 对应密集波形（高频）、NRZ=-1 对应稀疏波形（低频）：

In [ ]:
# 从相位直接生成 GFSK 调制波形，上方叠加 NRZ 做对比
fc = 0.5  
n_show = min(len(phase), 240)
t = np.arange(n_show)
n_bits_show = n_show // sps
gfsk_signal = np.cos(2 * np.pi * fc * t / sps + phase[:n_show])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5))

# 上图: NRZ (按采样点展开)
nrz_upsampled = np.repeat(nrz[:n_bits_show], sps)
ax1.step(np.arange(n_show), nrz_upsampled, "b-", where="post", lw=1.5)
ax1.set_xlim(0, n_show); ax1.set_ylim(-1.5, 1.5)
ax1.set_ylabel("NRZ"); ax1.set_title("NRZ Mapping")
ax1.grid(True, ls="--", alpha=0.3)
for b in range(0, n_bits_show + 1):
    ax1.axvline(x=b*sps, color="gray", ls="--", alpha=0.4, lw=0.5)

# 下图: GFSK 调制波形
ax2.plot(t, gfsk_signal, "b-", lw=0.6)
ax2.set_xlabel("Sample Index"); ax2.set_ylabel("Amplitude")
ax2.set_xlim(0, n_show); ax2.set_ylim(-1.3, 1.3)
ax2.set_title("GFSK Modulated Signal: fc=0.5")
ax2.grid(True, ls="--", alpha=0.3)
for b in range(0, n_bits_show + 1):
    ax2.axvline(x=b*sps, color="gray", ls="--", alpha=0.4, lw=0.5)

plt.tight_layout()
plt.show()

**步骤 5: 复指数映射 -> IQ 信号**

$e^{j*phase}$ 转为复基带 IQ 信号：

In [ ]:
iq_signal = np.exp(1j * phase)
fig,ax = plt.subplots(figsize=(12,3))
t = np.arange(len(iq_signal))
ax.plot(t,np.real(iq_signal),'b-',alpha=0.7,lw=0.8,label='I')
ax.plot(t,np.imag(iq_signal),'r-',alpha=0.7,lw=0.8,label='Q')
ax.set_xlabel('Sample Index');ax.set_xlim(0,len(iq_signal)-1)
ax.set_ylabel('Amplitude');ax.set_ylim(-1.2,1.2)
ax.set_title('Step 4: IQ Signal')
ax.legend();ax.grid(True,ls='--',alpha=0.5);plt.show()

以上就是 GFSKModulator.modulate() 的内部实现。实际使用时调用函数即可：

In [ ]:
mod = GFSKModulator(sps=8,mod_index=0.5)
tx_signal = mod.modulate(tx_bits)
print('调制完成')

---

### 5. AWGN 信道

加性高斯白噪声信道（AWGN，Additive White Gaussian Noise）是通信系统最经典最常用的信道模型。举例来说，发报机发送一系列信号，经过铜缆在接收机恢复出信号，就是一个典型的 AWGN 信道。接收端由于器件本身的电子随机运动会产生热噪声，接收信号总会被噪声污染，AWGN 信道模型可以很好地对此类热噪声进行建模，信道模型如下图：

<img src="./images/awgn.png" width="800">

AWGN 包含三个部分，每个都有特定含义：

- **A（Additive，加性）**：噪声与发送信号之间是相加的关系，$r(t) = s(t) + n(t)$。加性噪声一般是由于接收端低噪声放大器中的分子热运动造成的。还有一种由于本地晶振和接收信号相位失配造成的乘性噪声，在中低频（< 7 GHz）影响不大，在基础系统设计中通常只考虑加性噪声。

- **W（White，白）**：可以认为噪声 $n(t)$ 是一个各态历经的随机过程，其自相关函数为 $\delta$ 函数，功率谱在频率上没有起伏，在所有频率上均匀分布——如同白光包含所有颜色的光，因此称为"白色噪声"。

<img src="./images/awgn_power.png" width="600">

- **G（Gaussian，高斯）**：噪声 $n(t)$ 服从复高斯分布。由于热噪声由分子级别的运动引起，噪声取值是许多随机因素的叠加，按照中心极限定理，噪声总是服从高斯分布的。给定 $t$，热噪声 $n(t)$ 为复高斯变量，均值为零，方差为 $\sigma^2$。结合白噪声假设，不同时刻的噪声之间相互独立。

**SNR**（Signal-to-Noise Ratio，信噪比）定义为信号功率与噪声功率的比值：

$$\text{SNR(dB)} = 10 \cdot \log_{10}\left(\frac{P_{\text{signal}}}{P_{\text{noise}}}\right)$$

`ChannelModel.apply_awgn()` 的内部实现分三步：

$$\text{1. 测量信号功率： } P_{\text{signal}} = \frac{1}{N}\sum_{i=1}^{N} |s_i|^2$$

$$\text{2. 根据 SNR 计算噪声功率： } P_{\text{noise}} = \frac{P_{\text{signal}}}{10^{\text{SNR(dB)}/10}}$$

$$\text{3. 生成复高斯噪声并叠加： } r_i = s_i + \sqrt{\frac{P_{\text{noise}}}{2}} \cdot (n_I + j \cdot n_Q)$$

其中 $n_I, n_Q \sim \mathcal{N}(0, 1)$ 为独立的标准正态随机变量。除以 2 是因为复噪声的总功率需均分到 I/Q 两路。

下面拆解展示每一步，本实验选 SNR = 8 dB：

In [ ]:
from nearlink_sdr.phy.channel import ChannelModel
snr_db = 8.0

# Step 1: 测量信号功率
signal_power = np.mean(np.abs(tx_signal) ** 2)

# Step 2: 根据 SNR 计算噪声功率
snr_linear = 10.0 ** (snr_db / 10.0)
noise_power = signal_power / snr_linear

# Step 3: 生成高斯噪声并叠加
rng = np.random.default_rng(42)
noise = np.sqrt(noise_power / 2) * (rng.standard_normal(len(tx_signal)) + 1j * rng.standard_normal(len(tx_signal)))
rx_signal = tx_signal + noise

#Step 4：对比加噪前后的 IQ 信号的实部
n_show = 200
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5))

ax1.plot(np.arange(n_show), np.real(tx_signal[:n_show]), "b-", lw=0.6)
ax1.set_title("Before AWGN (clean)"); ax1.set_ylim(-1.5, 1.5)
ax1.set_ylabel("Amplitude"); ax1.grid(True, ls="--", alpha=0.3)

ax2.plot(np.arange(n_show), np.real(rx_signal[:n_show]), "r-", lw=0.6)
ax2.set_title(f"After AWGN (SNR={snr_db:.0f} dB)"); ax2.set_ylim(-1.5, 1.5)
ax2.set_xlabel("Sample Index"); ax2.set_ylabel("Amplitude")
ax2.grid(True, ls="--", alpha=0.3)

plt.tight_layout()
plt.show()
print("上图展示了 I 路信号加噪前后的对比")

以上就是给信号加入加性高斯白噪声的具体仿真实现，实际使用时可直接调用 apply_awgn() 函数。

---

### 6. GFSK 解调

GFSK 采用**频率鉴别器**进行非相干解调，无需载波相位同步。核心思路：频率高时相邻采样点相位差为正，频率低时相位差为负。

解调器代码如下所示：

```python
class GFSKDemodulator:
    """GFSK非相干解调器（基于频率鉴别器）。"""

    def __init__(self, sps: int = 8):
        self.sps = sps

    def demodulate(self, signal: np.ndarray) -> np.ndarray:
        """GFSK解调。

        :param signal: 复基带IQ信号

        :returns: 解调后的比特序列, 值为 0/1
        """
        # 频率鉴别：取相邻采样点的相位差
        phase_diff = np.angle(signal[1:] * np.conj(signal[:-1]))

        # 按符号累积频率 (包含末尾不足一个完整符号的部分)
        n_symbols = -(-len(phase_diff) // self.sps)  # ceiling division
        bits = np.zeros(n_symbols, dtype=int)
        for i in range(n_symbols):
            start = i * self.sps
            end = min(start + self.sps, len(phase_diff))
            segment = phase_diff[start:end]
            bits[i] = 1 if np.sum(segment) > 0 else 0

        return bits
```

`GFSKDemodulator.demodulate()` 分两步：

1. **提取瞬时频率**：计算相邻采样点的相位差，`phase_diff = angle(signal[i] × conj(signal[i-1]))`
2. **按符号判决**：每 sps 个相位差累加，和 > 0 判为比特 1（高频），< 0 判为比特 0（低频）

下面用加噪后的 `rx_signal` 来具体演示：

In [ ]:
# Step 1: 提取瞬时频率 —— 相邻采样点相位差
phase_diff = np.angle(rx_signal[1:] * np.conj(rx_signal[:-1]))

# Step 2: 按符号累积判决
n_symbols = -(-len(phase_diff) // sps)
rx_bits_manual = np.zeros(n_symbols, dtype=int)
for i in range(min(n_symbols, 10)):
    start = i * sps
    end = min(start + sps, len(phase_diff))
    seg_sum = np.sum(phase_diff[start:end])
    rx_bits_manual[i] = 1 if seg_sum > 0 else 0

# 仿真时直接调用 GFSKDemodulator.demodulate() 即可
from nearlink_sdr.phy.gfsk import GFSKDemodulator
demod = GFSKDemodulator(sps=8)
rx_bits = demod.demodulate(rx_signal)
print("解调完毕")

---

### 7. 误码统计

将解调后的比特与原始发送比特逐位比对：BER = 错误比特数 / 总传输比特数；BER 是衡量数字通信系统最核心的性能指标。

In [ ]:
n = min(len(tx_bits),len(rx_bits))
errors = int(np.sum(tx_bits[:n] != rx_bits[:n]))
ber = errors / n
print(f"SNR = 8 dB | {n} bits | {errors} errors | BER = {ber:.4e}")

---

## 课后实践

请补全下方 GFSK 手动调制与解调流程中的 **4 处空缺**（每处一行代码），在 SNR=8 dB 下对 100 帧数据完成调制→AWGN→解调的完整链路，并统计误码率。

要求：

1. **调制（2 处）**：补全 NRZ 映射和 IQ 信号生成
2. **解调（2 处）**：补全频率鉴别和符号判决

完成后运行 `python gfsk_practice.py`。

In [ ]:
%%writefile gfsk_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.phy.gfsk import GFSKModulator, GFSKDemodulator
from nearlink_sdr.phy.channel import ChannelModel

sps = 8
mod_index = 0.5
snr_db = 8.0
n_frames = 100
bits_per_frame = 100

rng = np.random.default_rng(42)
mod = GFSKModulator(sps=sps, mod_index=mod_index)
demod = GFSKDemodulator(sps=sps)
ch = ChannelModel(snr_db=snr_db)

total_bits = 0
total_errors = 0

for frame_idx in range(n_frames):
    tx_bits = rng.integers(0, 2, bits_per_frame)

    # ========== 调制 ==========
    # Step 1: NRZ 映射 —— 比特 0→-1, 比特 1→+1 （补全）
    nrz = 
    upsampled = np.repeat(nrz, sps)
    # 高斯滤波 + 相位积分
    filtered = np.convolve(upsampled, mod._gauss_filter, mode="same")
    freq_dev = mod_index * np.pi / sps
    phase = np.cumsum(filtered) * freq_dev
    # Step 2: 复指数映射 → IQ 信号 （补全）
    tx_signal = 

    # AWGN 信道
    rx_signal = ch.apply_awgn(tx_signal)

    # ========== 解调 ==========
    # Step 3: 频率鉴别 —— 相邻采样点共轭相乘取相位差 （补全）
    phase_diff = 
    # 按符号累积判决
    n_syms = -(-len(phase_diff) // sps)
    rx_bits = np.zeros(n_syms, dtype=int)
    for i in range(n_syms):
        start = i * sps
        end = min(start + sps, len(phase_diff))
        segment = phase_diff[start:end]
        # Step 4: 符号判决 —— 和>0 判为 1，和≤0 判为 0 （补全）
        rx_bits[i] = 

    n = min(len(tx_bits), len(rx_bits))
    total_bits += n
    total_errors += np.sum(tx_bits[:n] != rx_bits[:n])

ber = total_errors / total_bits
print(f"SNR={snr_db:.0f} dB | {n_frames} frames | {total_bits} bits |"
      f" {total_errors} errors | BER={ber:.4e}")


执行以下命令进行编译并验证结果：

In [ ]:
!python gfsk_practice.py

执行以下代码获取答案


In [ ]:
!cat answer/01.02_answer.txt
